# Persona Rater v2
## Open-Competition Design: All Variants Rated Per Product

### Key Changes from v1
- Each persona sees **all 5 variants** (GENERIC, E_PLUS, E_MINUS, O_PLUS, O_MINUS) for every product.
- No more MATCHED/MISMATCHED triplet structure — no assumption about what the 'opposite' message is.
- No axis blocks or order codes A–F — counterbalancing is handled purely by shuffling variant order per participant per product.
- 30 trials per persona (6 products × 5 variants), down from 36.
- After each product block, a **forced ranking** step is added (optional but recommended).
- Hypothesis tested at analysis stage: E_PLUS should be rated/ranked highest by HIGH_E personas.

### Pipeline Steps
1. Stimulus indexer + validator
2. Canonical index (p1 per variant)
3. Persona templates
4. Participant registry
5. Design schedules (30 trials, all-variants-per-product)
6. Rater runner (ratings + forced ranking)

## Step 1: Stimulus Indexer + Validator

In [2]:
import json
from pathlib import Path
from collections import defaultdict

# =========================
# CONFIG
# =========================
STIMULI_DIR = Path("stimuli_out")       # folder containing your generated stimulus JSONs
OUTPUT_DIR  = Path("design_out_v2")     # all design outputs go here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PRODUCTS = [f"P{i}" for i in range(1, 7)]
EXPECTED_VARIANTS = ["GENERIC", "E_PLUS", "E_MINUS", "O_PLUS", "O_MINUS"]

# =========================
# HELPERS
# =========================
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def extract_stimulus_fields(obj: dict, path: Path) -> dict:
    try:
        stimulus_id  = obj["id"]["output_id"]
        condition_id = obj["id"]["condition_id"]
        meta         = obj["meta"]
        product_id   = meta["product_id"]
        product_name = meta.get("product_name")
        trait_axis   = meta.get("trait_axis")
        variant      = meta["variant"]
        paraphrase_id = meta.get("paraphrase_id")
    except KeyError as e:
        raise KeyError(f"Missing key {e} in {path}")

    return {
        "stimulus_id":   stimulus_id,
        "condition_id":  condition_id,
        "filepath":      str(path.as_posix()),
        "product_id":    product_id,
        "product_name":  product_name,
        "trait_axis":    trait_axis,
        "variant":       variant,
        "paraphrase_id": paraphrase_id,
        "generation_model":         obj.get("generation", {}).get("model"),
        "generation_temperature":   obj.get("generation", {}).get("temperature"),
        "generation_timestamp_utc": obj.get("generation", {}).get("timestamp_utc"),
    }

# =========================
# SCAN + INDEX
# =========================
if not STIMULI_DIR.exists():
    raise FileNotFoundError(f"STIMULI_DIR not found: {STIMULI_DIR.resolve()}")

files = sorted(STIMULI_DIR.glob("*.json"))
print(f"Found {len(files)} JSON files in {STIMULI_DIR.resolve()}")

records = []
errors  = []

for fp in files:
    try:
        obj = load_json(fp)
        rec = extract_stimulus_fields(obj, fp)
        records.append(rec)
    except Exception as e:
        errors.append((str(fp), str(e)))

if errors:
    print("\n=== ERRORS ===")
    for fp, msg in errors[:20]:
        print(f"  - {fp}: {msg}")

# Build (product_id, variant) index
index_pv = defaultdict(list)
for r in records:
    index_pv[(r["product_id"], r["variant"])].append(r)

# =========================
# VALIDATION REPORT
# =========================
print("\n=== VALIDATION: coverage per product/variant ===")
missing    = []
duplicates = []

for pid in EXPECTED_PRODUCTS:
    counts = {var: len(index_pv.get((pid, var), [])) for var in EXPECTED_VARIANTS}
    counts_str = "  ".join([f"{var}:{counts[var]}" for var in EXPECTED_VARIANTS])
    print(f"  {pid}: {counts_str}")
    for var in EXPECTED_VARIANTS:
        n = counts[var]
        if n == 0:
            missing.append((pid, var))
        elif n > 1:
            duplicates.append((pid, var, n))

if missing:
    print("\n=== MISSING ===")
    for pid, var in missing:
        print(f"  - {pid} missing {var}")

if duplicates:
    print("\n=== DUPLICATES (will use p1 in Step 2) ===")
    for pid, var, n in duplicates:
        print(f"  - {pid} has {n} files for {var}")

# Save raw index
index_out = {
    "meta": {
        "stimuli_dir":        str(STIMULI_DIR.resolve().as_posix()),
        "n_files":            len(files),
        "n_parsed":           len(records),
        "n_errors":           len(errors),
        "expected_products":  EXPECTED_PRODUCTS,
        "expected_variants":  EXPECTED_VARIANTS,
    },
    "records": records,
}

out_path = OUTPUT_DIR / "stimuli_index.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(index_out, f, ensure_ascii=False, indent=2)

print(f"\nSaved index to: {out_path.resolve()}")
print("Next: Step 2 will select canonical p1 stimuli.")

Found 78 JSON files in C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\stimuli_out

=== VALIDATION: coverage per product/variant ===
  P1: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
  P2: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
  P3: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
  P4: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
  P5: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
  P6: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3

=== DUPLICATES (will use p1 in Step 2) ===
  - P1 has 3 files for E_PLUS
  - P1 has 3 files for E_MINUS
  - P1 has 3 files for O_PLUS
  - P1 has 3 files for O_MINUS
  - P2 has 3 files for E_PLUS
  - P2 has 3 files for E_MINUS
  - P2 has 3 files for O_PLUS
  - P2 has 3 files for O_MINUS
  - P3 has 3 files for E_PLUS
  - P3 has 3 files for E_MINUS
  - P3 has 3 files for O_PLUS
  - P3 has 3 files for O_MINUS
  - P4 has 

## Step 2: Canonical Index (p1 per variant)

In [15]:
import json, re
from pathlib import Path
from collections import defaultdict

INDEX_PATH = OUTPUT_DIR / "stimuli_index.json"
OUT_PATH   = OUTPUT_DIR / "stimuli_canonical_p1.json"

pat = re.compile(r"_p(\d+)_", re.IGNORECASE)

with INDEX_PATH.open("r", encoding="utf-8") as f:
    idx = json.load(f)

records = idx["records"]

def infer_style_id(filepath: str, variant: str) -> int:
    if variant == "GENERIC":
        return 0
    m = pat.search(filepath)
    return int(m.group(1)) if m else 0

for r in records:
    r["style_id"] = infer_style_id(r["filepath"], r["variant"])

# Group by (product_id, variant)
groups = defaultdict(list)
for r in records:
    groups[(r["product_id"], r["variant"])].append(r)

# Select canonical: GENERIC -> style_id=0, others -> style_id=1 (p1)
canonical = {}
missing   = []

for pid in EXPECTED_PRODUCTS:
    for var in EXPECTED_VARIANTS:
        key      = (pid, var)
        recs     = groups.get(key, [])
        target   = 0 if var == "GENERIC" else 1
        matches  = [r for r in recs if r["style_id"] == target]

        if not matches:
            missing.append(f"{pid}::{var} (target style_id={target})")
            canonical[f"{pid}::{var}"] = None
        elif len(matches) > 1:
            # Tie-break: alphabetical filepath
            canonical[f"{pid}::{var}"] = sorted(matches, key=lambda r: r["filepath"])[0]
        else:
            canonical[f"{pid}::{var}"] = matches[0]

# Validation report
print("=== Canonical selection (product | variants) ===")
for pid in EXPECTED_PRODUCTS:
    parts = []
    for var in EXPECTED_VARIANTS:
        rec = canonical.get(f"{pid}::{var}")
        if rec:
            parts.append(f"{var}:p{rec['style_id']} ({rec['stimulus_id'][:6]}...)")
        else:
            parts.append(f"{var}:MISSING")
    print(f"  {pid} | " + " | ".join(parts))

if missing:
    print("\n=== Problems ===")
    for m in missing:
        print(" -", m)
    raise RuntimeError("Fix missing stimuli before continuing.")

out = {
    "meta": {
        "version":      "canonical_p1_v2",
        "source_index": str(INDEX_PATH.as_posix()),
        "policy":       "GENERIC -> style_id=0; all non-generic variants -> style_id=1 (p1).",
        "variants":     EXPECTED_VARIANTS,
        "products":     EXPECTED_PRODUCTS,
    },
    "canonical_index": canonical
}

with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f"\nSaved canonical index to: {OUT_PATH.resolve()}")

=== Canonical selection (product | variants) ===
  P1 | GENERIC:p0 (f5d207...) | E_PLUS:p1 (3371e9...) | E_MINUS:p1 (8834de...) | O_PLUS:p1 (ebcf95...) | O_MINUS:p1 (722bbe...)
  P2 | GENERIC:p0 (bcc1a3...) | E_PLUS:p1 (d1f8bf...) | E_MINUS:p1 (e5ce5b...) | O_PLUS:p1 (ca789f...) | O_MINUS:p1 (e47d04...)
  P3 | GENERIC:p0 (93a62b...) | E_PLUS:p1 (988014...) | E_MINUS:p1 (8983db...) | O_PLUS:p1 (3ad25c...) | O_MINUS:p1 (b16dc8...)
  P4 | GENERIC:p0 (b582b4...) | E_PLUS:p1 (f1dd44...) | E_MINUS:p1 (6d8077...) | O_PLUS:p1 (bd0720...) | O_MINUS:p1 (2638c9...)
  P5 | GENERIC:p0 (e9bc51...) | E_PLUS:p1 (0395ba...) | E_MINUS:p1 (48595b...) | O_PLUS:p1 (bf3e4b...) | O_MINUS:p1 (082071...)
  P6 | GENERIC:p0 (d543ae...) | E_PLUS:p1 (d6f326...) | E_MINUS:p1 (cdf42c...) | O_PLUS:p1 (045746...) | O_MINUS:p1 (291f76...)

Saved canonical index to: C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\design_out

## Step 3: Persona Templates

Two persona types: HIGH_E and HIGH_O.
Each template includes a persona card and the rating questions.

In [16]:
import json
from pathlib import Path
from datetime import datetime, timezone

TEMPLATE_DIR = Path("persona_templates")
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)

def utc_now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

TEMPLATES = [
    {
        "type_id": "TYPE_E_HIGH",
        "big5":    {"O": 50, "C": 50, "E": 70, "A": 50, "N": 50},
        "persona_card": {
            "about_me": (
                "I tend to prefer messages that feel lively, direct, and socially engaging. "
                "I react positively to energy and confidence, and I like when an ad feels "
                "easy to share or talk about with others."
            ),
            "how_i_judge_ads": [
                "I respond well to upbeat, confident framing.",
                "I prefer a sense of momentum and social energy.",
                "If something feels dull, overly cautious, or solitary, I am less persuaded."
            ],
            "response_style_rules": [
                "When asked for ratings, I choose a number from 1 to 7.",
                "When asked for a ranking, I order ads from most to least persuasive.",
                "When asked for motivation, I provide exactly one sentence.",
                "I do not rewrite the ad text."
            ]
        }
    },
    {
        "type_id": "TYPE_O_HIGH",
        "big5":    {"O": 70, "C": 50, "E": 50, "A": 50, "N": 50},
        "persona_card": {
            "about_me": (
                "I tend to prefer messages that feel interesting, imaginative, and thought-provoking. "
                "I like novel angles and vivid detail, and I respond well when an ad feels original "
                "rather than generic."
            ),
            "how_i_judge_ads": [
                "I respond well to novelty, creativity, and distinctive phrasing.",
                "I prefer ads that spark curiosity or suggest a unique experience.",
                "If something feels bland, formulaic, or purely practical, I am less persuaded."
            ],
            "response_style_rules": [
                "When asked for ratings, I choose a number from 1 to 7.",
                "When asked for a ranking, I order ads from most to least persuasive.",
                "When asked for motivation, I provide exactly one sentence.",
                "I do not rewrite the ad text."
            ]
        }
    }
]

# Rating questions — same two questions as v1
RATER_APPENDIX = {
    "q1_text":      "I find this ad to be persuasive",
    "q2_text":      "This ad has made me more interested in [product]",
    "scale_anchors": {"1": "strongly disagree", "7": "strongly agree"},
    "output_contract": {
        "effectiveness_rating_q1": "integer 1-7",
        "effectiveness_rating_q2": "integer 1-7",
        "motivation": "exactly one sentence"
    }
}

COMMON = {
    "version":        "2.0",
    "created_at_utc": utc_now_iso(),
    "thresholds":     {"HIGH_if_ge": 60, "LOW_if_le": 40, "MID_otherwise": True},
    "rater_instructions_appendix": RATER_APPENDIX
}

written = []
for t in TEMPLATES:
    out  = {**COMMON, **t}
    path = TEMPLATE_DIR / f"{t['type_id']}.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    written.append(path)

print("Wrote persona templates:")
for p in written:
    print(" -", p.resolve())

Wrote persona templates:
 - C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\persona_templates\TYPE_E_HIGH.json
 - C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\persona_templates\TYPE_O_HIGH.json


## Step 4: Participant Registry

100 participants per persona type (200 total).
Each participant gets a unique respondent_seed used to shuffle variant order in Step 5.
No axis_order_group needed — that concept is removed in v2.

In [17]:
import json, random
from pathlib import Path
from datetime import datetime, timezone

PART_DIR = Path("participants_out")
PART_DIR.mkdir(parents=True, exist_ok=True)

N_PER_TYPE  = 100
MASTER_SEED = 20260303  # fixed for reproducibility
rng = random.Random(MASTER_SEED)

def utc_now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

REGISTRIES = [
    ("TYPE_E_HIGH", "PER_EHIGH"),
    ("TYPE_O_HIGH", "PER_OHIGH"),
]

written = []
for type_id, prefix in REGISTRIES:
    participants = []
    for i in range(1, N_PER_TYPE + 1):
        persona_id      = f"{prefix}_{i:04d}"
        respondent_seed = rng.randrange(1, 2**31 - 1)
        participants.append({
            "persona_id":      persona_id,
            "type_id":         type_id,
            "respondent_seed": respondent_seed,
        })

    out = {
        "type_id":          type_id,
        "n_participants":   N_PER_TYPE,
        "generated_at_utc": utc_now_iso(),
        "master_seed":      MASTER_SEED,
        "participants":     participants
    }
    path = PART_DIR / f"{type_id}_participants.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    written.append(path)

print("Wrote participant registries:")
for p in written:
    print(" -", p.resolve())
print(f"\nTotal participants: {len(REGISTRIES) * N_PER_TYPE}")
print("Trials per participant: 6 products × 5 variants = 30")

Wrote participant registries:
 - C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\participants_out\TYPE_E_HIGH_participants.json
 - C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\participants_out\TYPE_O_HIGH_participants.json

Total participants: 200
Trials per participant: 6 products × 5 variants = 30


## Step 5: Design Schedules

Each participant sees all 5 variants for each product (30 trials total).
- Products are shuffled per participant using respondent_seed.
- Within each product block, variant order is also shuffled using respondent_seed.
- This ensures no variant is systematically presented first across participants.

No MATCHED/MISMATCHED roles. No axis blocks. No order codes.

**Trial structure per participant:**
```
For each product (shuffled order):
    For each variant in [GENERIC, E_PLUS, E_MINUS, O_PLUS, O_MINUS] (shuffled order):
        Present ad → collect Q1 rating, Q2 rating, motivation
    After all 5 variants of product:
        Forced ranking of the 5 ads just seen
```

In [18]:
import json, random
from pathlib import Path
from datetime import datetime, timezone

CANON_PATH = OUTPUT_DIR / "stimuli_canonical_p1.json"
PART_DIR   = Path("participants_out")
DESIGN_DIR = OUTPUT_DIR

PRODUCTS = [f"P{i}" for i in range(1, 7)]
VARIANTS = ["GENERIC", "E_PLUS", "E_MINUS", "O_PLUS", "O_MINUS"]

def utc_now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

# Load canonical index
canon = load_json(CANON_PATH)
canonical_index = canon["canonical_index"]

def canon_get(product_id, variant):
    key = f"{product_id}::{variant}"
    if key not in canonical_index or canonical_index[key] is None:
        raise KeyError(f"Missing canonical stimulus: {key}")
    return canonical_index[key]

def make_design_for_participant(p: dict) -> dict:
    persona_id      = p["persona_id"]
    type_id         = p["type_id"]
    respondent_seed = p["respondent_seed"]
    rng             = random.Random(respondent_seed)

    # Shuffle product order for this participant
    product_order = PRODUCTS[:]
    rng.shuffle(product_order)

    trials         = []
    trial_global   = 0
    product_blocks = []  # for ranking step metadata

    for product_id in product_order:
        # Shuffle variant order within this product block
        variant_order = VARIANTS[:]
        rng.shuffle(variant_order)

        block_trial_ids = []

        for within_block_index, variant in enumerate(variant_order, start=1):
            trial_global += 1
            rec = canon_get(product_id, variant)

            trial = {
                "trial_index_global":      trial_global,
                "trial_index_within_block": within_block_index,
                "product_id":              product_id,
                "product_name":            rec.get("product_name"),
                "variant":                 variant,
                "stimulus_id":             rec["stimulus_id"],
                "stimulus_condition_id":   rec["condition_id"],
                "stimulus_filepath":       rec["filepath"],
                # Label for within-product forced ranking
                "block_label":             f"{product_id}_block",
            }
            trials.append(trial)
            block_trial_ids.append(trial_global)

        product_blocks.append({
            "product_id":         product_id,
            "product_name":       canon_get(product_id, "GENERIC").get("product_name"),
            "variant_order":      variant_order,
            "trial_indices":      block_trial_ids,
            "ranking_prompt":     (
                f"You have just rated 5 ads for '{canon_get(product_id, 'GENERIC').get('product_name')}'. "
                f"Please rank them from most to least persuasive (1 = most persuasive, 5 = least persuasive). "
                f"Return ONLY a JSON object with keys: "
                + str({f'rank_{v}': 'integer 1-5' for v in variant_order})
            )
        })

    return {
        "persona_id":      persona_id,
        "type_id":         type_id,
        "respondent_seed": respondent_seed,
        "created_at_utc":  utc_now_iso(),
        "design_version":  "v2_open_competition",
        "n_trials":        trial_global,
        "n_products":      len(PRODUCTS),
        "n_variants":      len(VARIANTS),
        "source": {
            "canonical_index": str(CANON_PATH.as_posix()),
            "policy": "All 5 variants rated per product. No matched/mismatched roles. "
                       "Variant and product order shuffled per participant via respondent_seed."
        },
        "trials":          trials,
        "product_blocks":  product_blocks,
    }

# Load all participant registries and generate designs
part_files = sorted(PART_DIR.glob("TYPE_*_participants.json"))
if not part_files:
    raise FileNotFoundError(f"No participant registries found in {PART_DIR.resolve()}")

participants_all = []
for pf in part_files:
    obj = load_json(pf)
    participants_all.extend(obj["participants"])

print(f"Loaded {len(participants_all)} participants.")

written = 0
for p in participants_all:
    design   = make_design_for_participant(p)
    out_path = DESIGN_DIR / f"{p['persona_id']}_design.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(design, f, ensure_ascii=False, indent=2)
    written += 1

print(f"Wrote {written} design files to: {DESIGN_DIR.resolve()}")

# Sanity check on one design file
sample_path = DESIGN_DIR / f"{participants_all[0]['persona_id']}_design.json"
with sample_path.open("r", encoding="utf-8") as f:
    d = load_json(sample_path)

print(f"\nSample design: {d['persona_id']}")
print(f"Total trials: {d['n_trials']}")
by_variant = {}
for t in d["trials"]:
    by_variant[t["variant"]] = by_variant.get(t["variant"], 0) + 1
print("Trials by variant:", by_variant)
print("Product blocks:", len(d["product_blocks"]))
print("\nFirst product block order:", d["product_blocks"][0]["variant_order"])

Loaded 200 participants.
Wrote 200 design files to: C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\design_out_v2

Sample design: PER_EHIGH_0001
Total trials: 30
Trials by variant: {'O_MINUS': 6, 'GENERIC': 6, 'E_MINUS': 6, 'E_PLUS': 6, 'O_PLUS': 6}
Product blocks: 6

First product block order: ['O_MINUS', 'GENERIC', 'E_MINUS', 'E_PLUS', 'O_PLUS']


## Step 6: Rater Runner

For each participant:
1. For each trial: present ad → collect Q1 rating, Q2 rating, motivation
2. After each product block (5 ads): collect forced ranking of those 5 ads

**Expected output per trial:**
```json
{"effectiveness_rating_q1": 5, "effectiveness_rating_q2": 4, "motivation": "One sentence."}
```

**Expected output per ranking:**
```json
{"rank_GENERIC": 3, "rank_E_PLUS": 1, "rank_E_MINUS": 5, "rank_O_PLUS": 2, "rank_O_MINUS": 4}
```

**Note:** `rate_one()` and `rank_block()` are stubs — replace with your real API call.

In [20]:
import json, re, os
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from openai import OpenAI

# =========================
# CONFIG
# =========================
DESIGN_DIR   = OUTPUT_DIR
TEMPLATE_DIR = Path("persona_templates")
OUT_DIR      = Path("persona_ratings_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load .env file first, before accessing any environment variables
load_dotenv()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

RUN_ID      = "RUN_v2_001"   # update per run
MODEL_NAME  = "gpt-4o"
TEMPERATURE = 0.7            # higher than v1 to simulate realistic inter-individual variance

MESSAGE_CANDIDATE_PATHS = [
    ("content", "full_message"),
    ("content", "message"),
    ("content", "text"),
]

# =========================
# HELPERS
# =========================
def utc_now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def get_message_text(stimulus_obj: dict, filepath: str) -> str:
    for path in MESSAGE_CANDIDATE_PATHS:
        cur = stimulus_obj
        ok  = True
        for k in path:
            if isinstance(cur, dict) and k in cur:
                cur = cur[k]
            else:
                ok = False
                break
        if ok and isinstance(cur, str) and cur.strip():
            return cur.strip()
    raise KeyError(f"Could not find message text in {filepath}.")

def one_sentence(s: str) -> bool:
    terminals = re.findall(r"[.!?]", s.strip())
    return len(terminals) <= 1

def validate_rating_output(out: dict):
    for k in ["effectiveness_rating_q1", "effectiveness_rating_q2"]:
        if k not in out:
            raise ValueError(f"Missing key: {k}")
        if not isinstance(out[k], int) or not (1 <= out[k] <= 7):
            raise ValueError(f"{k} must be int 1–7, got {out[k]}")
    if not isinstance(out.get("motivation"), str):
        raise ValueError("Missing/invalid motivation")
    if not one_sentence(out["motivation"]):
        raise ValueError(f"Motivation not one-sentence: {out['motivation']!r}")

def validate_ranking_output(out: dict, variants: list):
    expected_keys = {f"rank_{v}" for v in variants}
    if set(out.keys()) != expected_keys:
        raise ValueError(f"Ranking keys mismatch. Expected {expected_keys}, got {set(out.keys())}")
    ranks = list(out.values())
    if sorted(ranks) != list(range(1, len(variants) + 1)):
        raise ValueError(f"Ranks must be a permutation of 1–{len(variants)}, got {ranks}")

def call_api(prompt: str) -> str:
    """Shared API call used by both rate_one and rank_block."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        response_format={"type": "json_object"},  # forces JSON output
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content.strip()
# =========================
# PROMPT BUILDERS
# =========================
def build_rating_prompt(persona_template: dict, message_text: str, product_name: str) -> str:
    q1      = persona_template["rater_instructions_appendix"]["q1_text"]
    q2      = persona_template["rater_instructions_appendix"]["q2_text"].replace("[product]", product_name or "the product")
    anchors = persona_template["rater_instructions_appendix"]["scale_anchors"]
    card    = persona_template["persona_card"]
    about   = card["about_me"]
    rules   = "\n".join([f"- {x}" for x in card["response_style_rules"]])

    return f"""You are a survey respondent.

Respondent description:
{about}

Response rules:
{rules}

Ad to rate (do not rewrite it):
\"\"\"{message_text}\"\"\"

Rate the following statements on a 1–7 scale ({anchors['1']} … {anchors['7']}):
Q1: {q1}
Q2: {q2}

Return ONLY a JSON object with:
- effectiveness_rating_q1 (int 1-7)
- effectiveness_rating_q2 (int 1-7)
- motivation (exactly one sentence explaining your rating)
"""

def build_ranking_prompt(persona_template: dict, product_name: str,
                          ads: list, variant_order: list) -> str:
    """
    ads: list of (variant, message_text) in the order they were presented.
    Returns a prompt asking the persona to rank all 5 ads.
    """
    card  = persona_template["persona_card"]
    about = card["about_me"]
    rules = "\n".join([f"- {x}" for x in card["response_style_rules"]])

    ads_block = "\n\n".join([
        f"Ad {i+1} (ID: {variant}):\n\"\"\"{text}\"\"\""
        for i, (variant, text) in enumerate(ads)
    ])

    rank_keys = str({f'rank_{v}': 'integer 1-5 (1=most persuasive)' for v in variant_order})

    return f"""You are a survey respondent.

Respondent description:
{about}

Response rules:
{rules}

You have just rated the following 5 ads for '{product_name}':

{ads_block}

Now rank all 5 ads from most to least persuasive (1 = most persuasive, 5 = least persuasive).
Each ad must receive a unique rank.

Return ONLY a JSON object with these keys:
{rank_keys}
"""

# =========================
# API STUBS (REPLACE THESE)
# =========================
def rate_one(prompt: str) -> dict:
    """
    Calls the API and returns:
    {effectiveness_rating_q1: int, effectiveness_rating_q2: int, motivation: str}
    """
    raw = call_api(prompt)
    try:
        out = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"API returned invalid JSON: {raw}") from e

    # Coerce ratings to int in case the model returns strings
    out["effectiveness_rating_q1"] = int(out["effectiveness_rating_q1"])
    out["effectiveness_rating_q2"] = int(out["effectiveness_rating_q2"])
    return out

def rank_block(prompt: str) -> dict:
    """
    Calls the API and returns:
    {rank_GENERIC: int, rank_E_PLUS: int, rank_E_MINUS: int,
     rank_O_PLUS: int, rank_O_MINUS: int} — a permutation of 1–5.
    """
    raw = call_api(prompt)
    try:
        out = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"API returned invalid JSON: {raw}") from e

    # Coerce all rank values to int
    out = {k: int(v) for k, v in out.items()}
    return out

# =========================
# RUNNER
# =========================
design_files = sorted(DESIGN_DIR.glob("PER_*_design.json"))
print(f"Found {len(design_files)} design files in {DESIGN_DIR.resolve()}")

if not design_files:
    raise FileNotFoundError("No design files found. Run Step 5 first.")

MAX_PARTICIPANTS_TO_RUN = None  # set to None to run all

ran_participants = 0

for design_path in design_files:
    if MAX_PARTICIPANTS_TO_RUN is not None and ran_participants >= MAX_PARTICIPANTS_TO_RUN:
        break

    design        = load_json(design_path)
    persona_id    = design["persona_id"]
    type_id       = design["type_id"]
    template_path = TEMPLATE_DIR / f"{type_id}.json"
    persona_template = load_json(template_path)

    out_persona_dir = OUT_DIR / persona_id
    out_persona_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Running {persona_id} ({type_id}) ===")

    # Group trials by product block
    blocks_by_product = {b["product_id"]: b for b in design["product_blocks"]}
    trials_by_product = {}
    for t in design["trials"]:
        pid = t["product_id"]
        if pid not in trials_by_product:
            trials_by_product[pid] = []
        trials_by_product[pid].append(t)

    # Iterate over products in the order defined by the design
    product_order = [b["product_id"] for b in design["product_blocks"]]

    for product_id in product_order:
        block          = blocks_by_product[product_id]
        product_name   = block["product_name"]
        variant_order  = block["variant_order"]
        product_trials = trials_by_product[product_id]

        ads_seen = []  # accumulate (variant, message_text) for ranking step

        # --- Individual trial ratings ---
        for t in product_trials:
            trial_idx  = t["trial_index_global"]
            stim_fp    = Path(t["stimulus_filepath"])
            stim_obj   = load_json(stim_fp)
            msg_text   = get_message_text(stim_obj, str(stim_fp))
            variant    = t["variant"]

            rating_prompt = build_rating_prompt(persona_template, msg_text, product_name)
            rating_output = rate_one(rating_prompt)  # <-- replace stub
            validate_rating_output(rating_output)

            ads_seen.append((variant, msg_text))

            # Save trial rating artifact
            rating_artifact = {
                "meta": {
                    "persona_id":             persona_id,
                    "type_id":                type_id,
                    "trial_index_global":     trial_idx,
                    "trial_index_within_block": t["trial_index_within_block"],
                    "product_id":             product_id,
                    "product_name":           product_name,
                    "variant":                variant,
                    "stimulus_id":            t["stimulus_id"],
                    "stimulus_condition_id":  t["stimulus_condition_id"],
                    "stimulus_filepath":      t["stimulus_filepath"],
                    "model":       {"name": MODEL_NAME, "temperature": TEMPERATURE},
                    "run":         {"run_id": RUN_ID, "timestamp_utc": utc_now_iso()}
                },
                "questions": {
                    "q1_text":      persona_template["rater_instructions_appendix"]["q1_text"],
                    "q2_text":      persona_template["rater_instructions_appendix"]["q2_text"],
                    "scale_anchors": persona_template["rater_instructions_appendix"]["scale_anchors"]
                },
                "output": rating_output
            }

            out_path = out_persona_dir / f"trial_{trial_idx:04d}__stim_{t['stimulus_id']}.json"
            with out_path.open("w", encoding="utf-8") as f:
                json.dump(rating_artifact, f, ensure_ascii=False, indent=2)

        # --- Forced ranking after all 5 ads for this product ---
        ranking_prompt = build_ranking_prompt(
            persona_template, product_name, ads_seen, variant_order
        )
        ranking_output = rank_block(ranking_prompt)  # <-- replace stub
        validate_ranking_output(ranking_output, variant_order)

        # Save ranking artifact
        ranking_artifact = {
            "meta": {
                "persona_id":    persona_id,
                "type_id":       type_id,
                "product_id":    product_id,
                "product_name":  product_name,
                "variant_order": variant_order,
                "model":         {"name": MODEL_NAME, "temperature": TEMPERATURE},
                "run":           {"run_id": RUN_ID, "timestamp_utc": utc_now_iso()}
            },
            "output": ranking_output
        }

        rank_path = out_persona_dir / f"ranking__{product_id}.json"
        with rank_path.open("w", encoding="utf-8") as f:
            json.dump(ranking_artifact, f, ensure_ascii=False, indent=2)

        print(f"  {product_id}: rated 5 ads + ranked. Top: {min(ranking_output, key=ranking_output.get)}")

    print(f"  Saved to: {out_persona_dir.resolve()}")
    ran_participants += 1

print("\nRunner finished (stub mode).")

Found 200 design files in C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\design_out_v2

=== Running PER_EHIGH_0001 (TYPE_E_HIGH) ===
  P1: rated 5 ads + ranked. Top: rank_E_PLUS
  P4: rated 5 ads + ranked. Top: rank_E_PLUS
  P2: rated 5 ads + ranked. Top: rank_E_PLUS
  P3: rated 5 ads + ranked. Top: rank_E_PLUS
  P6: rated 5 ads + ranked. Top: rank_E_PLUS
  P5: rated 5 ads + ranked. Top: rank_E_PLUS
  Saved to: C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\AI_personality_persuasion-1\Stimulus_Builder\persona_ratings_out\PER_EHIGH_0001

=== Running PER_EHIGH_0002 (TYPE_E_HIGH) ===
  P5: rated 5 ads + ranked. Top: rank_E_PLUS
  P3: rated 5 ads + ranked. Top: rank_E_PLUS
  P4: rated 5 ads + ranked. Top: rank_E_PLUS
  P6: rated 5 ads + ranked. Top: rank_E_PLUS
  P1: rated 5 ads + ranked. Top: rank_E_PLUS
  P2: rated 5 ads + ranked. Top: r

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## Step 7: Analysis Helper

After ratings and rankings are collected, this cell loads all results and computes:
- Mean Q1/Q2 rating per variant per persona type
- Rank frequency: how often each variant ranked #1 per persona type
- Effect size (Cohen's d) for the target variant vs all others

**Core hypothesis tests:**
- HIGH_E personas: E_PLUS rated/ranked higher than all other variants
- HIGH_O personas: O_PLUS rated/ranked higher than all other variants

In [13]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

RATINGS_DIR = Path("persona_ratings_out")
VARIANTS    = ["GENERIC", "E_PLUS", "E_MINUS", "O_PLUS", "O_MINUS"]

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

# =========================
# LOAD RATINGS
# =========================
rating_rows  = []
ranking_rows = []

for persona_dir in sorted(RATINGS_DIR.iterdir()):
    if not persona_dir.is_dir():
        continue
    persona_id = persona_dir.name

    for fp in sorted(persona_dir.glob("trial_*.json")):
        obj = load_json(fp)
        m   = obj["meta"]
        o   = obj["output"]
        rating_rows.append({
            "persona_id":  persona_id,
            "type_id":     m["type_id"],
            "product_id":  m["product_id"],
            "variant":     m["variant"],
            "q1":          o["effectiveness_rating_q1"],
            "q2":          o["effectiveness_rating_q2"],
            "motivation":  o["motivation"],
        })

    for fp in sorted(persona_dir.glob("ranking__*.json")):
        obj = load_json(fp)
        m   = obj["meta"]
        o   = obj["output"]
        row = {
            "persona_id": persona_id,
            "type_id":    m["type_id"],
            "product_id": m["product_id"],
        }
        for variant in VARIANTS:
            row[f"rank_{variant}"] = o.get(f"rank_{variant}")
        ranking_rows.append(row)

ratings_df  = pd.DataFrame(rating_rows)
rankings_df = pd.DataFrame(ranking_rows)

print(f"Rating rows loaded:  {len(ratings_df)}")
print(f"Ranking rows loaded: {len(rankings_df)}")

# =========================
# MEAN RATINGS PER TYPE × VARIANT
# =========================
print("\n=== Mean Q1+Q2 ratings per persona type × variant ===")
ratings_df["q_mean"] = (ratings_df["q1"] + ratings_df["q2"]) / 2
mean_table = ratings_df.groupby(["type_id", "variant"])["q_mean"].agg(["mean", "std", "count"])
print(mean_table.round(3).to_string())

# =========================
# RANK-1 FREQUENCY PER TYPE × VARIANT
# =========================
print("\n=== Rank-1 frequency (most persuasive) per persona type × variant ===")
rank_rows = []
for _, row in rankings_df.iterrows():
    for variant in VARIANTS:
        rank_rows.append({
            "type_id":  row["type_id"],
            "variant":  variant,
            "rank":     row[f"rank_{variant}"],
            "is_rank1": row[f"rank_{variant}"] == 1
        })

rank_df    = pd.DataFrame(rank_rows)
rank1_freq = rank_df.groupby(["type_id", "variant"])["is_rank1"].agg(["sum", "mean", "count"])
rank1_freq.columns = ["rank1_count", "rank1_rate", "n"]
print(rank1_freq.round(3).to_string())

# =========================
# COHEN'S D: TARGET VARIANT VS ALL OTHERS
# =========================
def cohens_d(x, y):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna().astype(float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").dropna().astype(float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    sp = np.sqrt(((len(x)-1)*x.std(ddof=1)**2 + (len(y)-1)*y.std(ddof=1)**2) / (len(x)+len(y)-2))
    return (x.mean() - y.mean()) / sp if sp > 0 else np.nan

HYPOTHESES = [
    ("TYPE_E_HIGH", "E_PLUS"),
    ("TYPE_O_HIGH", "O_PLUS"),
]

print("\n=== Cohen's d: target variant vs all other variants (q_mean) ===")
for type_id, target_variant in HYPOTHESES:
    sub    = ratings_df[ratings_df["type_id"] == type_id]
    target = sub[sub["variant"] == target_variant]["q_mean"]
    others = sub[sub["variant"] != target_variant]["q_mean"]
    d      = cohens_d(target, others)
    print(f"  {type_id} | {target_variant} vs rest: d = {d:.3f} "
          f"(mean_target={target.mean():.2f}, mean_others={others.mean():.2f})")

Rating rows loaded:  60
Ranking rows loaded: 12

=== Mean Q1+Q2 ratings per persona type × variant ===
                      mean    std  count
type_id     variant                     
TYPE_E_HIGH E_MINUS  3.833  0.516      6
            E_PLUS   6.250  0.612      6
            GENERIC  5.583  0.665      6
            O_MINUS  4.417  0.665      6
            O_PLUS   5.750  0.274      6
TYPE_O_HIGH E_MINUS  5.667  0.408      6
            E_PLUS   6.250  0.274      6
            GENERIC  5.250  0.612      6
            O_MINUS  3.750  0.612      6
            O_PLUS   6.417  0.204      6

=== Rank-1 frequency (most persuasive) per persona type × variant ===
                     rank1_count  rank1_rate  n
type_id     variant                            
TYPE_E_HIGH E_MINUS            0         0.0  6
            E_PLUS             6         1.0  6
            GENERIC            0         0.0  6
            O_MINUS            0         0.0  6
            O_PLUS             0         0.0  